# NB03 — Server Resource Utilization

Profiles **CPU usage**, **memory (RAM)**, and estimates **Redis cache hit impact**
for each major QCanvas operation.

**Smart Monitoring:** This notebook will automatically detect if you are running the backend via `uvicorn` natively and track its PID. If not found (e.g., running in Docker), it will fall back to monitoring the notebook process itself.

**Requires:** Backend running at `http://localhost:8000`  
**Output:** `../results/resources/cpu_mem.csv`, `../results/resources/cache_stats.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
from pathlib import Path

from api_timing import get_auth_token, auth_headers
from resource_monitor import (
    ResourceMonitor, profile_request, profile_n_requests,
    measure_python_memory, system_snapshot, find_process_by_name
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})

RESULTS_DIR = Path('../results/resources')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'http://localhost:8000'
N_RUNS = 5   # runs per operation for averaging

# --- AUTO-DETECT SERVER FOR MONITORING ---
SERVER_PID = find_process_by_name("uvicorn")
if SERVER_PID:
    print(f'✅ Found uvicorn backend (PID: {SERVER_PID}). Monitoring SERVER resources.')
else:
    print('⚠  Uvicorn process not found. Monitoring CLIENT resources (notebook overhead) instead.')
# ------------------------------------------

print('✓ Imports OK')

## 0 — System Snapshot

In [ ]:
snap = system_snapshot()
print('System information:')
for k, v in snap.items():
    print(f'  {k}: {v}')

In [ ]:
token = get_auth_token()
hdrs = auth_headers(token)
print(f'Auth: {"✓" if token else "⚠ none"}')

## 1 — CPU + Memory per Framework Conversion

Profile the converter endpoint for all three frameworks.

In [ ]:
CONVERT_URL = f'{BASE_URL}/api/converter/convert'

payloads = [
    ('Cirq Bell', {
        'code': 'import cirq\nq0, q1 = cirq.LineQubit.range(2)\ncircuit = cirq.Circuit([cirq.H(q0), cirq.CNOT(q0, q1), cirq.measure(q0, q1, key="r")])\n',
        'framework': 'cirq',
    }),
    ('Qiskit Grover', {
        'code': 'from qiskit import QuantumCircuit\nqc = QuantumCircuit(2, 2)\nqc.h([0, 1])\nqc.cz(0, 1)\nqc.h([0, 1])\nqc.x([0, 1])\nqc.cz(0, 1)\nqc.x([0, 1])\nqc.h([0, 1])\nqc.measure([0, 1], [0, 1])',
        'framework': 'qiskit',
    }),
    ('PennyLane VQE', {
        'code': 'import pennylane as qml\nimport numpy as np\ndev = qml.device("default.qubit", wires=2)\n@qml.qnode(dev)\ndef vqe_circuit(theta):\n    qml.Hadamard(wires=0)\n    qml.RX(theta[0], wires=0)\n    qml.RY(theta[1], wires=1)\n    qml.CNOT(wires=[0, 1])\n    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))',
        'framework': 'pennylane',
    }),
]

cpu_mem_rows = []

for label, payload in payloads:
    print(f'Profiling: {label} ({N_RUNS} runs)...')
    profiles = profile_n_requests('POST', CONVERT_URL, n=N_RUNS, headers=hdrs, json=payload, label=label, target_pid=SERVER_PID)
    row = {
        'operation': label,
        'type': 'conversion',
        'mean_latency_ms': sum(p.request_latency_ms for p in profiles) / len(profiles),
        'mean_peak_cpu_pct': sum(p.peak_cpu for p in profiles) / len(profiles),
        'mean_peak_rss_mb': sum(p.peak_rss_mb for p in profiles) / len(profiles),
        'max_peak_rss_mb': max(p.peak_rss_mb for p in profiles),
    }
    cpu_mem_rows.append(row)
    print(f'  latency={row["mean_latency_ms"]:.0f}ms  peak_cpu={row["mean_peak_cpu_pct"]:.1f}%  peak_ram={row["mean_peak_rss_mb"]:.0f}MB')

## 2 — CPU + Memory per Simulation Backend

In [ ]:
SIM_URL = f'{BASE_URL}/api/simulator/simulate'
QASM_2Q = 'OPENQASM 3.0;\nqubit[2] q;\nbit[2] c;\nh q[0];\ncx q[0], q[1];\nc[0] = measure q[0];\nc[1] = measure q[1];\n'

sim_configs = [
    ('Statevector 2q 512 shots', {'qasm_code': QASM_2Q, 'backend': 'statevector', 'shots': 512}),
    ('Statevector 2q 4096 shots', {'qasm_code': QASM_2Q, 'backend': 'statevector', 'shots': 4096}),
    ('Density Matrix 2q 512 shots', {'qasm_code': QASM_2Q, 'backend': 'density_matrix', 'shots': 512}),
]

for label, payload in sim_configs:
    print(f'Profiling: {label} ({N_RUNS} runs)...')
    profiles = profile_n_requests('POST', SIM_URL, n=N_RUNS, headers=hdrs, json=payload, label=label, target_pid=SERVER_PID)
    row = {
        'operation': label,
        'type': 'simulation',
        'mean_latency_ms': sum(p.request_latency_ms for p in profiles) / len(profiles),
        'mean_peak_cpu_pct': sum(p.peak_cpu for p in profiles) / len(profiles),
        'mean_peak_rss_mb': sum(p.peak_rss_mb for p in profiles) / len(profiles),
        'max_peak_rss_mb': max(p.peak_rss_mb for p in profiles),
    }
    cpu_mem_rows.append(row)
    print(f'  latency={row["mean_latency_ms"]:.0f}ms  peak_cpu={row["mean_peak_cpu_pct"]:.1f}%  peak_ram={row["mean_peak_rss_mb"]:.0f}MB')

## 3 — Save and Plot Resource Usage

In [ ]:
df = pd.DataFrame(cpu_mem_rows)
df.to_csv(RESULTS_DIR / 'cpu_mem.csv', index=False)
print('✓ Saved cpu_mem.csv')
display(df.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue' if t == 'conversion' else 'orchid' for t in df['type']]
short_ops = [op.replace(' shots', 'sh').replace('Statevector', 'SV').replace('Density Matrix', 'DM') for op in df['operation']]

# Latency
axes[0].bar(short_ops, df['mean_latency_ms'], color=colors, alpha=0.85)
axes[0].set_title('Mean Latency per Operation')
axes[0].set_ylabel('Latency (ms)')
axes[0].tick_params(axis='x', rotation=35)

# Peak CPU
axes[1].bar(short_ops, df['mean_peak_cpu_pct'], color=colors, alpha=0.85)
axes[1].axhline(80, color='red', linestyle='--', linewidth=1, label='80% warn')
axes[1].set_title('Mean Peak CPU %')
axes[1].set_ylabel('CPU (%)')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend()

# Peak RAM
axes[2].bar(short_ops, df['mean_peak_rss_mb'], color=colors, alpha=0.85)
axes[2].set_title('Mean Peak RSS (MB)')
axes[2].set_ylabel('RAM (MB)')
axes[2].tick_params(axis='x', rotation=35)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Conversion'), Patch(facecolor='orchid', label='Simulation')]
fig.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cpu_mem_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved cpu_mem_chart.png')

## 4 — Redis Cache Hit Rate Estimation

We estimate cache effectiveness by sending identical requests multiple times
and comparing first-hit (cold) vs. subsequent hit (warm) latency.
A significant speedup implies cache hits.

In [ ]:
import httpx, time, statistics

def estimate_cache_hit_rate(url, method, payload, headers, n_repeats=10):
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        r = httpx.request(method, url, json=payload, headers=headers, timeout=60)
        times.append((time.perf_counter() - t0) * 1000)
    cold = times[0]
    warm = statistics.mean(times[1:])
    speedup = cold / warm if warm > 0 else 1.0
    # Heuristic: speedup > 1.5x suggests caching
    cache_likely = speedup > 1.5
    return {
        'cold_ms': cold, 'warm_mean_ms': warm, 'speedup': speedup,
        'cache_likely': cache_likely, 'all_times': times,
    }

cache_tests = [
    ('Converter (same Cirq circuit)', 'POST', f'{BASE_URL}/api/converter/convert',
     {'code': 'import cirq\nq0, q1 = cirq.LineQubit.range(2)\ncircuit = cirq.Circuit([cirq.H(q0), cirq.CNOT(q0, q1), cirq.measure(q0, q1, key="r")])\n', 'framework': 'cirq'}),
    ('Simulator (same QASM)', 'POST', f'{BASE_URL}/api/simulator/simulate',
     {'qasm_code': QASM_2Q, 'backend': 'statevector', 'shots': 512}),
]

cache_rows = []
for label, method, url, payload in cache_tests:
    print(f'Cache test: {label}...')
    res = estimate_cache_hit_rate(url, method, payload, hdrs, n_repeats=10)
    r = {'label': label, 'cold_ms': res['cold_ms'], 'warm_mean_ms': res['warm_mean_ms'],
         'speedup': res['speedup'], 'cache_likely': res['cache_likely']}
    cache_rows.append(r)
    print(f'  cold={r["cold_ms"]:.0f}ms  warm={r["warm_mean_ms"]:.0f}ms  speedup={r["speedup"]:.2f}x  cache={r["cache_likely"]}')

df_cache = pd.DataFrame(cache_rows)
df_cache.to_csv(RESULTS_DIR / 'cache_stats.csv', index=False)
print('\n✓ Saved cache_stats.csv')
display(df_cache.round(2))